# TO-Agents — run the demo

A group of AI agents turns a written description of a structural problem into a
validated setup, runs a 3D topology optimization, looks at the rendered result
with a vision model, proposes revisions, and scores the candidates.

**To start:** add an API key to Secrets (the 🔑 icon on the left), then
**Runtime → Run all**. It takes about four minutes and prints a link.

| key | cost | where |
|---|---|---|
| `GEMINI_API_KEY` | free, no card | https://aistudio.google.com/apikey |
| `OPENROUTER_API_KEY` | ~$0.002 per run | https://openrouter.ai/keys |

Either one works. Give the notebook access to it with the toggle beside it.


In [ ]:
#@title  { display-mode: "form" }
# Everything happens in quickstart.py so this stays one cell.
import os, subprocess, sys

REPO_DIR = '/content/to-agents-lite'
REPO_URL = 'https://github.com/bellastewart/to-agents-lite.git'

# --- key, from Colab Secrets --------------------------------------------------
_found = []
try:
    from google.colab import userdata
    for _canonical, _aliases in {
        'GEMINI_API_KEY':     ['GEMINI_API_KEY', 'GEMINI', 'Gemini', 'GOOGLE_API_KEY'],
        'OPENROUTER_API_KEY': ['OPENROUTER_API_KEY', 'OPENROUTER', 'Openrouter',
                               'OpenRouter', 'openrouter'],
    }.items():
        for _a in _aliases:
            try:
                _v = userdata.get(_a)
            except Exception:
                _v = None
            if _v:
                os.environ[_canonical] = _v.strip()
                _found.append(_canonical)
                break
except Exception:
    pass

if not _found:
    import getpass
    _v = getpass.getpass('Paste a Gemini or OpenRouter API key (then press Enter): ').strip()
    if not _v:
        raise SystemExit(
            'No key given. Add GEMINI_API_KEY or OPENROUTER_API_KEY to Secrets '
            '(the key icon on the left) and run this cell again.')
    os.environ['OPENROUTER_API_KEY' if _v.startswith('sk-or') else 'GEMINI_API_KEY'] = _v

# --- code ---------------------------------------------------------------------
# fetch + reset, not pull: this must land on origin/main whatever the local
# state is. A plain `git pull` can fail (diverged history, a stray file) and
# with captured output that failure is INVISIBLE -- you then keep running old
# code while believing you updated, which is a genuinely awful way to debug.
if os.path.isdir(REPO_DIR + '/.git'):
    _r = subprocess.run(['git', '-C', REPO_DIR, 'fetch', '-q', 'origin', 'main'],
                        capture_output=True, text=True)
    if _r.returncode == 0:
        _r = subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', '-q', 'origin/main'],
                            capture_output=True, text=True)
else:
    _r = subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_DIR],
                        capture_output=True, text=True)
if _r.returncode != 0:
    print('WARNING: could not update the code — continuing with whatever is on disk.')
    print((_r.stderr or _r.stdout).strip()[:300])
else:
    _v = subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h %s'],
                        capture_output=True, text=True).stdout.strip()
    print(f'code: {_v[:72]}')

# --- everything else ----------------------------------------------------------
# Stream the child's output line by line rather than subprocess.run().
#
# A subprocess inherits the kernel's REAL stdout (file descriptor 1), but
# IPython captures cell output by swapping sys.stdout at the Python level. So
# subprocess.run() prints nothing here -- the progress lines and the URL go to
# the kernel log where nobody sees them, and the cell shows only the returned
# CompletedProcess repr. Piping and re-printing puts it back in the cell, and
# has the side benefit of appearing live instead of all at the end.
import re as _re
_url = None
_p = subprocess.Popen([sys.executable, 'quickstart.py'], cwd=REPO_DIR,
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1)
for _line in _p.stdout:
    print(_line, end='')
    _m = _re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', _line)
    if _m and _url is None:
        _url = _m.group(0)
_p.wait()

if _p.returncode != 0:
    print('\nSetup did not finish. The lines above say why.')
elif _url:
    # Open it for them, and render a large link as the fallback. Browsers block
    # window.open when it is not tied to a click, and this one is fired by a
    # cell finishing, so it is blocked often enough that the link has to be the
    # real interface -- the auto-open is a convenience on top, never the only
    # way through.
    from IPython.display import display, HTML
    display(HTML(f'''
    <div style="margin:18px 0;padding:20px 22px;border:1px solid #e6e4df;
                border-radius:10px;background:#f6f6f3;
                font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif">
      <div style="font-size:13px;letter-spacing:.08em;text-transform:uppercase;
                  color:#9a9a97;margin-bottom:10px">Your session is ready</div>
      <a href="{_url}" target="_blank" rel="noopener"
         style="display:inline-block;background:#141414;color:#fff;text-decoration:none;
                padding:12px 22px;border-radius:8px;font-weight:600;font-size:16px">
        Open the app &rarr;</a>
      <div style="margin-top:12px;font-size:13px;color:#6b6b6b">
        Opening automatically. If nothing happens, use the button
        &mdash; some browsers block automatic tabs.<br>
        <code style="font-size:12px">{_url}</code>
      </div>
    </div>
    <script>setTimeout(function(){{ window.open("{_url}", "_blank"); }}, 1200);</script>
    '''))
None
